# Bronze layer — ERPNext bank GL ingestion

1. Load local configuration and credentials from `data-platform/.env`.
2. Write and read two synthetic Delta rows in MinIO under `bronze/_smoke/`.
3. Read the single bank-side GL entry for the configured demo journal voucher.
4. Land the original ERPNext response bytes under `bronze/raw/erpnext/gl_entry/`.
5. Merge the source row into Bronze Delta and check replay without duplicates.

Place this notebook in `data-platform/src/data_platform/reconciliation/bronze/`. Use a Python 3.11 kernel and JDK 17 for this Spark 3.5 setup. In that environment install:

```powershell
python -m pip install pyspark==3.5.6 python-dotenv requests "boto3>=1.36,<2"
```

Keep `data-platform/.env` Git-ignored. Reuse its `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY`, then add the following values locally:

| Variable | Value |
|---|---|
| `ERPNEXT_URL` | ERPNext site base URL you use in the browser |
| `ERPNEXT_API_KEY`, `ERPNEXT_API_SECRET` | API credentials for an identity with read access to the target GL Entry |
| `ERPNEXT_COMPANY` | Exact demo company name |
| `ERPNEXT_VOUCHER_NO` | Submitted Journal Entry ID |
| `ERPNEXT_BANK_ACCOUNT` | Exact bank ledger account on the credit line |
| `MINIO_ENDPOINT` | Optional. Defaults to `http://192.168.29.250:9000` |
| `BRONZE_BUCKET` | Optional. Defaults to `bronze`, which must already exist |

Use only synthetic data for the local HTTP POC. For shared environments, use TLS and a storage identity scoped to the ingestion paths. Run from a fresh kernel and run the cells in order. Each full run creates a fresh smoke-test table.

In [ ]:
import hashlib
import json
import os
import sys
from decimal import Decimal
from pathlib import Path
from urllib.parse import urlsplit
from uuid import uuid4

import boto3
import requests
from botocore.config import Config
from botocore.exceptions import ClientError
from dotenv import dotenv_values
from pyspark.sql import SparkSession, functions as F

# Support launching Jupyter from either the repository root or the notebook folder.
cwd = Path.cwd().resolve()
candidates = [parent if parent.name == "data-platform" else parent / "data-platform"
              for parent in (cwd, *cwd.parents)]
env_path = next((folder / ".env" for folder in candidates if (folder / ".env").is_file()), None)
if env_path is None:
    raise FileNotFoundError("Create data-platform/.env and launch from the repository.")
cfg = dotenv_values(env_path, interpolate=False)

def required(name):
    value = cfg.get(name)
    if not value:
        raise ValueError(f"Set {name} in data-platform/.env")
    return value

endpoint = cfg.get("MINIO_ENDPOINT") or "http://192.168.29.250:9000"
bucket = cfg.get("BRONZE_BUCKET") or "bronze"
os.environ["AWS_ACCESS_KEY_ID"] = required("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = required("AWS_SECRET_ACCESS_KEY")
os.environ["PYSPARK_PYTHON"] = sys.executable

# Use path-style addressing for both Spark and the raw-object client.
s3 = boto3.client(
    "s3", endpoint_url=endpoint, region_name="us-east-1",
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    config=Config(s3={"addressing_style": "path"}),
)
spark = (
    SparkSession.builder.master("local[2]").appName("bronze-erpnext-gl")
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.3.3,org.apache.hadoop:hadoop-aws:3.3.4")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.hadoop.fs.s3a.endpoint", endpoint)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", str(urlsplit(endpoint).scheme == "https").lower())
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)

## 1. Verify the MinIO Delta write and read

In [ ]:
# Isolate the test objects from the Bronze data.
smoke_path = f"s3a://{bucket}/_smoke/delta-{uuid4().hex}"
spark.range(2).write.format("delta").mode("errorifexists").save(smoke_path)
ids = [row.id for row in spark.read.format("delta").load(smoke_path).orderBy("id").collect()]
if ids != [0, 1]:
    raise RuntimeError("Unexpected Delta smoke-test result.")
print("PASS: wrote and read 2 Delta rows")

## 2. Extract the bank-side GL entry

In [ ]:
site = required("ERPNEXT_URL").rstrip("/")
company = required("ERPNEXT_COMPANY")
voucher = required("ERPNEXT_VOUCHER_NO")
account = required("ERPNEXT_BANK_ACCOUNT")

# Fetch at most two rows so an ambiguous match is detected before writing.
response = requests.get(
    f"{site}/api/resource/GL%20Entry",
    headers={"Authorization": f"token {required('ERPNEXT_API_KEY')}:{required('ERPNEXT_API_SECRET')}"},
    params={
        "fields": json.dumps(["name", "company", "voucher_type", "voucher_no",
                              "account", "posting_date", "debit", "credit", "is_cancelled"]),
        "filters": json.dumps([
            ["company", "=", company], ["voucher_type", "=", "Journal Entry"],
            ["voucher_no", "=", voucher], ["account", "=", account],
            ["is_cancelled", "=", 0],
        ]),
        "limit_page_length": 2,
    },
    timeout=30,
    allow_redirects=False,
)
if response.status_code != 200:
    raise RuntimeError(f"ERPNext returned HTTP {response.status_code}. Check the site and API permissions.")
rows = response.json()["data"]
if len(rows) != 1:
    raise RuntimeError(f"Expected one bank GL row, received {len(rows)}.")
row = rows[0]
if not row.get("name") or Decimal(str(row["credit"])) != 10000 or Decimal(str(row["debit"])) != 0:
    raise RuntimeError("The response does not match the expected 10,000 INR bank credit.")
raw_bytes = response.content

## 3. Land raw JSON and merge into Bronze Delta

In [ ]:
# A source key identifies the logical row. A content hash identifies the raw capture.
source_key = hashlib.sha256(json.dumps([site, row["name"]]).encode()).hexdigest()
raw_hash = hashlib.sha256(raw_bytes).hexdigest()
raw_key = f"raw/erpnext/gl_entry/{source_key}/{raw_hash}.json"
raw_uri = f"s3a://{bucket}/{raw_key}"
table_uri = f"s3a://{bucket}/tables/erpnext/gl_entry"
run_id = uuid4().hex

def land_and_merge():
    try:
        s3.put_object(Bucket=bucket, Key=raw_key, Body=raw_bytes,
                      ContentType="application/json", IfNoneMatch="*")
    except ClientError as exc:
        if exc.response["ResponseMetadata"]["HTTPStatusCode"] != 412:
            raise

    # Keep the source fields and link each Bronze row to its exact raw response.
    frame = spark.read.json(spark.sparkContext.parallelize([json.dumps(row)], 1))
    frame = (frame.withColumn("source_key", F.lit(source_key))
             .withColumn("source_instance", F.lit(site))
             .withColumn("raw_uri", F.lit(raw_uri))
             .withColumn("run_id", F.lit(run_id))
             .withColumn("ingested_at", F.current_timestamp()))
    frame.createOrReplaceTempView("incoming_erpnext_gl")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS delta.`{table_uri}`
        USING DELTA AS SELECT * FROM incoming_erpnext_gl WHERE 1 = 0
    """)
    # Apply the latest capture for an existing key only when its content changes.
    spark.sql(f"""
        MERGE INTO delta.`{table_uri}` AS existing
        USING incoming_erpnext_gl AS incoming
        ON existing.source_key = incoming.source_key
        WHEN MATCHED AND existing.raw_uri <> incoming.raw_uri THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

land_and_merge()
print(f"Bronze write complete. Run {run_id}, rows 1, raw capture {raw_uri}")

## 4. Verify replay and lineage

Replay the captured response through the same write path. You can also rerun the notebook from a fresh kernel to test a second ERPNext fetch.

In [ ]:
land_and_merge()
with s3.get_object(Bucket=bucket, Key=raw_key)["Body"] as body:
    if body.read() != raw_bytes:
        raise RuntimeError("Raw MinIO content differs from the ERPNext response.")

# Count only the selected source record so the check remains valid as Bronze grows.
matches = (spark.read.format("delta").load(table_uri)
           .where(F.col("source_key") == source_key).select("raw_uri").take(2))
if len(matches) != 1 or matches[0].raw_uri != raw_uri:
    raise RuntimeError("Bronze replay or raw lineage verification failed.")
print("PASS: raw capture verified and one Bronze row after replay")